## 1. Basic Tasks

**1. Load the messy e-commerce dataset and identify columns containing nulls and duplicate rows using
.filter(), .distinct(), and .dropDuplicates().**

In [0]:
df = spark.read.csv(
    "/Volumes/dev/bronze/raw/messy_ecommerce_extended.csv",
    header=True,
    inferSchema=True    
)

In [0]:
from pyspark.sql import functions as F

null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])

null_counts.display()

In [0]:
from pyspark.sql.functions import *
distinct_df = df.distinct()
null_orders = df.filter(col("order_id").isNotNull())
cleaned_duplicates_orders = null_orders.dropDuplicates(["order_id"])
cleaned_orders = cleaned_duplicates_orders.dropna(how="any")
cleaned_orders.display()

**2. Rename at least 3 columns to a consistent naming convention (e.g., snake_case) using
withColumnRenamed.**

In [0]:
df_renamed = cleaned_orders.withColumnRenamed("customer_id", "customerId").withColumnRenamed("order_id", "orderId").withColumnRenamed("order_date", "orderDate").withColumnRenamed("product_category", "productCategory")
df_renamed.display()

**3. Connect a Databricks Repo to a Git provider and make your first commit of a cleaning notebook.**

For connecting a Databricks repo to GitHub we can do it two ways:
- first is create a repo on Github an then copy the the git repo link and create a git folder in DataBricks and paste the git repo link in DataBricks git folder and in your account and go to Linked account and configure the GitHub and add the add the repo for access. and then create a notbook and perform the tasks and then click on the git link and then put a commit and then click on commit & push.
- In the second way first create a GitHub repo then create a normal folder and in your account and go to Linked account and configure the GitHub and add the add the repo for access. Then click on the terminal and then run the below code part:
```
git init
git add .
git commit -m "first commit"
git branch -M main
git remote add origin https://github.com/SurajitM0nd0l/new_repo_name.git
git push -u origin main
```

## 2. Intermediate Tasks

**4. Build a cleaning pipeline: drop fully-null rows, fill remaining nulls with sensible defaults, remove
exact duplicates, and sort by order_date.**

In [0]:
df = spark.read.csv(
    "/Volumes/dev/bronze/raw/sales.csv",
    header = True,
    inferSchema = True
)

In [0]:
from pyspark.sql.functions import col, current_date, coalesce

cleaned_df = (
    df.dropna(how="all")
    .withColumn("order_date", coalesce(col("order_date"), current_date()))
    .fillna({
        "order_id": -1,
        "customer_id": -1,
        "transaction_id": -1,
        "product_id": -1,
        "quantity": 0,
        "discount_amount": 0,
        "total_amount": 0.00
    })
    .dropDuplicates()
    .orderBy("order_date")
)

In [0]:
df.write.mode("overwrite").saveAsTable("dev.silver.cleaned_sales_new")

**5. Perform an aggregation (revenue by category or region) and a join against a second small reference
table (e.g., customers or regions).**

In [0]:
from pyspark.sql.functions import sum

transactions = spark.read.csv(
    "/Volumes/dev/bronze/raw/transaction.csv", 
    header=True, 
    inferSchema=True
)
customers = spark.read.csv(
    "/Volumes/dev/bronze/raw/customers-3.csv", 
    header=True, 
    inferSchema=True
)

transactions_clean = transactions.dropDuplicates().dropna()
customers_clean = customers.dropDuplicates().dropna()

merged_df = transactions_clean.join(customers_clean, "customer_id", "left")

summary_df = merged_df.groupBy("status").agg(sum("amount").alias("total_amount"))

summary_df.write.mode("overwrite").saveAsTable("dev.gold.financial_summary")

**6. Create a feature branch in your Databricks Repo, change the cleaning logic, and open a pull request
describing what changed and why.**

I completed the task by following these steps:
1. Pushed the current pipeline code to the `main` branch to ensure the latest version was available in the repository.

2. Modified the pipeline's cleaning logic by changing:

   ```
   df.dropna(subset=["order_id", "order_date"])
   ```

   to:

   ```
   df.dropna(how="all")
   ```

3. Created a feature branch in Databricks from the `main` branch. I committed the changes to the feature branch with an appropriate commit message.

4. Created a Pull Request in GitHub from the feature branch to the `main` branch, describing the changes made to the pipeline.

5. Merged the Pull Request into the `main` branch after reviewing the changes.

6. Deleted the feature branch from GitHub after the Pull Request was successfully merged, as the changes were now part of the `main` branch.


## 3. Advanced Tasks

**7. Extend the pipeline to handle a 'dirty data' scenario not covered in class (e.g., mixed date formats,
currency symbols in numeric columns) and document your approach.**

In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

@dp.table
def sales_cleaned_extended():
    df = spark.read.csv(
        "/Volumes/dev/bronze/raw/sales-3.csv",
        header=True,
        inferSchema=True
    )

    df_cleaned = (
        df.dropna(how="all")
        
        # 1. Clean Identifiers: Remove whitespace and standardize case
        .withColumn("order_id", F.upper(F.trim(F.col("order_id"))))
        .withColumn("customer_id", F.upper(F.trim(F.col("customer_id"))))
        .withColumn("product_id", F.upper(F.trim(F.col("product_id"))))
        
        # 2. Clean Quantity: Handle textual numbers and cast to integer
        .withColumn("quantity", F.lower(F.trim(F.col("quantity"))))
        .withColumn("quantity", 
            F.when(F.col("quantity") == "one", "1")
             .when(F.col("quantity") == "two", "2")
             .when(F.col("quantity") == "three", "3")
             .when(F.col("quantity") == "four", "4")
             .when(F.col("quantity") == "five", "5")
             .otherwise(F.col("quantity"))
        )
        .withColumn("quantity", F.col("quantity").cast("integer"))
        
        # 3. Clean Financials: Use RegEx to strip currency symbols and text, then cast to double
        .withColumn("total_amount", F.regexp_replace(F.col("total_amount"), r"[^\d\.\-]", "").cast("double"))
        .withColumn("discount_amount", F.regexp_replace(F.col("discount_amount"), r"[^\d\.\-]", "").cast("double"))
        
        # 4. Clean Dates: Use coalesce to attempt multiple date formats
        .withColumn("order_date_parsed", F.coalesce(
            F.to_date(F.col("order_date"), "yyyy-MM-dd"),
            F.to_date(F.col("order_date"), "MM/dd/yyyy"),
            F.to_date(F.col("order_date"), "yyyy/MM/dd"),
            F.to_date(F.col("order_date"), "MM-dd-yyyy"),
            F.to_date(F.col("order_date"), "MMM d, yyyy"),
            F.to_date(F.col("order_date"), "dd-MM-yy")
        ))
        # Fallback to current_date() if the date was completely unparseable (e.g., "Unknown", "TBD")
        .withColumn("order_date", F.coalesce(F.col("order_date_parsed"), F.current_date()))
        .drop("order_date_parsed")

        # 5. Handle Nulls: Use string defaults for ID columns
        .fillna({
            "order_id": "UNKNOWN",
            "customer_id": "UNKNOWN",
            "transaction_id": "UNKNOWN",
            "product_id": "UNKNOWN",
            "quantity": 0,
            "discount_amount": 0.0,
            "total_amount": 0.0
        })
        .dropDuplicates()
        # Note: orderBy is generally not supported in readStream without a watermark, 
        # but it is kept here to match your original pipeline structure.
        .orderBy("order_date")
    )

    return df_cleaned

**8. Set up a branching strategy (dev/main) for the Cyntexa analytics repo and write a short guide for
teammates on the pull-request review workflow before merging into main.**

# Cyntexa Analytics Repository: Branching Strategy and Pull Request Workflow

## Branching Strategy

For the Cyntexa analytics repository, we will use two main branches:

* **`main`** – The stable and production-ready branch. Changes should only be merged into `main` after review and validation.
* **`dev`** – The development and integration branch. New features, pipeline changes, and cleaning logic should be tested here before being promoted to `main`.

For individual tasks, developers should create a short-lived feature branch from `dev`.

Example:

```text
main
  ↑
 dev
  ↑
feature/sales-cleaning
```

The recommended workflow is:

1. Create or switch to the latest `dev` branch.
2. Create a feature branch from `dev`, such as `feature/sales-cleaning`.
3. Make and test the required changes.
4. Commit the changes with a clear commit message.
5. Push the feature branch to GitHub.
6. Open a Pull Request from the feature branch into `dev`.
7. After review and successful testing, merge the Pull Request into `dev`.
8. When the changes in `dev` are ready for release, create a Pull Request from `dev` into `main`.
9. Review and validate the changes before merging them into `main`.

## Pull Request Review Workflow

Before merging any Pull Request into `main`, teammates should follow these steps:

1. **Review the description** – Confirm that the PR clearly explains what was changed and why.
2. **Review the code** – Check the pipeline logic, transformations, naming, formatting, and overall implementation.
3. **Check data quality** – Verify that the changes correctly handle expected and dirty data scenarios without unnecessarily removing valid records.
4. **Test the changes** – Run the affected Databricks notebooks or pipelines and confirm that they complete successfully.
5. **Check for unintended changes** – Make sure the PR contains only the files and changes relevant to the task.
6. **Leave comments when necessary** – Ask questions or request changes if something needs clarification or improvement.
7. **Approve the PR** – Once the implementation and tests are satisfactory, approve the Pull Request.
8. **Merge into `main`** – Merge only after the required reviews and checks have passed.

## Example Workflow

```text
dev
 │
 ├── feature/sales-cleaning
 │       │
 │       ├── Make changes
 │       ├── Test
 │       ├── Commit
 │       └── Push
 │
 │       ↓
 │   Pull Request
 │       ↓
 │   Code Review
 │       ↓
 │   Merge → dev
 │
 └───────────────
                 ↓
          Release-ready dev
                 ↓
          Pull Request
                 ↓
            Review + Tests
                 ↓
             Merge → main
```

This branching strategy keeps `main` stable while allowing developers to work independently and safely integrate changes through `dev`. The Pull Request review process provides an additional quality-control step before changes reach the production-ready `main` branch.


**9. (Data Analyst) Using the cleaned dataset, produce a summary report answering 3 business questions
(e.g., top-selling category per region, month-over-month growth, average order value trend) and
note any data-quality caveats a stakeholder should know about.**

In [0]:
%sql

-- 1. Month-over-Month (MoM) Revenue Growth

CREATE OR REPLACE VIEW dev.gold.sales_mom_growth AS

WITH MonthlyRevenue AS (
    SELECT DATE_TRUNC('month', sale_date) AS month, SUM(sale_amount) AS revenue
    FROM dev.silver.sales_clean
    GROUP BY DATE_TRUNC('month', sale_date)
),

RevenueWithLag AS (
    SELECT month, revenue, LAG(revenue) OVER (ORDER BY month) AS prev_revenue
    FROM MonthlyRevenue
)

SELECT month, revenue, prev_revenue, 
    ROUND(((revenue - prev_revenue) / NULLIF(prev_revenue, 0)) * 100,2) AS mom_growth_pct
FROM RevenueWithLag

ORDER BY month;

In [0]:
%sql
-- 2. Average Revenue Per User (ARPU) by Region
CREATE OR REPLACE VIEW dev.gold.sales_arpu_by_region AS

WITH RegionAggregates AS (
    SELECT region, SUM(sale_amount) AS total_revenue, COUNT(DISTINCT customer_id) AS unique_customers
    FROM dev.silver.sales_clean
    GROUP BY region
)

SELECT region, total_revenue, unique_customers, 
    ROUND(total_revenue / CAST(unique_customers AS FLOAT), 2) AS arpu
FROM RegionAggregates
ORDER BY arpu DESC;

In [0]:
%sql
-- 3. Customer Purchase Frequency by region
CREATE OR REPLACE VIEW dev.gold.sales_purchase_frequency AS

WITH OrderAggregates AS (
    SELECT region, COUNT(DISTINCT sale_id) AS total_orders, COUNT(DISTINCT customer_id) AS unique_customers
    FROM dev.silver.sales_clean
    GROUP BY region
)

SELECT region, total_orders, unique_customers, 
    ROUND(total_orders / CAST(unique_customers AS FLOAT), 2) AS purchase_frequency
FROM OrderAggregates;